In [47]:
import numpy as np
import time
import itertools

# DETERMINANTES

### Análisis de Complejidad

* **Complejidad Temporal:**
    *  $\mathcal{O}(n^3)$ (Cúbica). NumPy utiliza el método de descomposición LU (similar a la eliminación de Gauss) para calcular el determinante. Esto hace que el cálculo sea extremadamente rápido incluso con matrices grandes.
* **Complejidad Espacial:**
    * $\mathcal{O}(n^2)$ (Cuadrática). El programa solo consume memoria para almacenar la matriz bidimensional de $n \times n$ y las operaciones auxiliares del cálculo.
Usa el código con precaución.¿Necesitas que simplifiquemos el análisis de algún otro código de tu notebook?

In [48]:
#Introducir datos

tamaño = int(input('Ingrese el tamaño de la matriz'))

def determinante(tamaño):
    matriz = np.random.randint(1,100, size=(tamaño,tamaño))
    print(f'Matriz de {tamaño} * {tamaño}:\n{matriz} ')
    determinante = np.linalg.det(matriz)
    return f'Su determinante es: {determinante}'

inicio = time.time()
var = determinante(tamaño)
fin= time.time()
print(f'Tiempo de cálculo: {round(fin - inicio, 4)} segundos')
var

Ingrese el tamaño de la matriz 3


Matriz de 3 * 3:
[[70 57 65]
 [72 91  4]
 [31 48 38]] 
Tiempo de cálculo: 0.0014 segundos


'Su determinante es: 121011.00000000015'

# PERMANENTE: Manera 1

### Análisis de Complejidad (Permanente con Permutaciones)

* **Complejidad Temporal:**
    * $\mathcal{O}(n \cdot n!)$ (Factorial). El código genera todas las permutaciones posibles de las columnas ($n!$) y, por cada una, recorre las filas ($n$) para multiplicar los elementos. Es un algoritmo extremadamente lento para matrices mayores a 10x10.
* **Complejidad Espacial:**
    * $\mathcal{O}(n^2)$ (Cuadrática). Solo guarda en memoria la matriz original de $n \times n$. Las permutaciones se procesan una a una sin acumularse en la memoria RAM.


In [50]:
def calcular_permanente(matriz):
    n = len(matriz)
    permanente = 0
    # itertools.permutations genera las n! combinaciones de columnas
    for p in itertools.permutations(range(n)):
        producto_diagonal = 1
        for i in range(n):
            producto_diagonal *= matriz[i][p[i]]
        permanente += producto_diagonal
    return permanente

n = int(input('Ingrese el tamaño (n) para la matriz aleatoria n*n: '))

# Advertencia por la complejidad factorial O(n!)
if n > 10:
    print("⚠️ ¡Cuidado! El cálculo del permanente para matrices mayores a 10 puede tardar mucho o congelar tu PC.")
    confirmar = input("¿Deseas continuar de todos modos? (s/n): ")
    if confirmar.lower() != 's':
        n = 10
        print("Ajustado automáticamente a un tamaño seguro de 10x10.")

# 2. Generar matriz aleatoria con números enteros del 1 al 10
# (Usamos números pequeños para que el resultado final no sea un número gigantesco)
matriz_aleatoria = np.random.randint(1, 11, size=(n, n))
print(f'\nMatriz Aleatoria de {n} * {n}:\n{matriz_aleatoria}\n')

# 3. Calcular el permanente midiendo el tiempo de ejecución
inicio = time.time()
resultado_permanente = calcular_permanente(matriz_aleatoria)
fin = time.time()

print(f'El permanente es: {resultado_permanente}')
print(f'Tiempo de cálculo: {round(fin - inicio, 4)} segundos')


Ingrese el tamaño (n) para la matriz aleatoria n*n:  3



Matriz Aleatoria de 3 * 3:
[[ 1 10  8]
 [ 2  2  9]
 [ 1  3  5]]

El permanente es: 291
Tiempo de cálculo: 0.0001 segundos


# PERMANENTE: Manera 2 

### Análisis de Complejidad (Fórmula de Ryser)

* **Complejidad Temporal:**
    * $\mathcal{O}(n \cdot 2^n)$ (Exponencial). El algoritmo recorre todas las combinaciones posibles de subconjuntos de columnas ($2^n$). En cada iteración, realiza operaciones vectoriales con NumPy (`np.sum` y `np.prod`) sobre las filas de la matriz ($n$). Es drásticamente más rápido que el método factorial, permitiendo calcular matrices de hasta 15x15 en pocos segundos.
* **Complejidad Espacial:**
    * $\mathcal{O}(n^2)$ (Cuadrática). Mantiene en memoria RAM la matriz original de tamaño $n \times n$ y algunos arreglos unidimensionales temporales de tamaño $n$ para las sumas y máscaras de bits de cada ciclo.
Usa el código con precaución.¿Te gustaría que grafiquemos cómo se comportan estos tres métodos (Determinante, Permanente Factorial y Permanente Ryser) para ver la diferencia de tiempos en tu notebook?

In [46]:
import numpy as np
import time

def ryser_permanent(M):
    """
    Calcula el permanente de una matriz usando la fórmula de Ryser.
    Complejidad de tiempo: O(n * 2^n)
    """
    n = M.shape[0]
    perm = 0
    
    # Precalculamos las potencias de 2 para las operaciones de bits
    potencias_dos = 1 << np.arange(n)
    
    # Iteramos sobre los 2^n - 1 subconjuntos posibles (excluyendo el conjunto vacío)
    for i in range(1, 2**n):
        # Creamos una máscara booleana de las columnas activas en esta iteración
        mask = (i & potencias_dos) > 0
        
        # Sumamos las columnas seleccionadas por cada fila
        row_sums = np.sum(M[:, mask], axis=1)
        
        # Multiplicamos el resultado de las sumas de todas las filas
        term = np.prod(row_sums)
        
        # Calculamos el signo: (-1)^(n - número de columnas seleccionadas)
        sign = (-1)**(n - mask.sum())
        
        perm += sign * term
        
    return perm

def calcular_permanente_aleatorio():
    try:
        n = int(input("Ingrese el tamaño de la matriz cuadrada (ej. 3 para una de 3x3): "))
        if n <= 0:
            print("El tamaño de la matriz debe ser un número entero positivo mayor a 0.")
            return
            
        # Advertencia de seguridad algorítmica
        if n > 15:
            print(f"\n¡Advertencia! El cálculo del permanente para una matriz de {n}x{n} es exponencial.")
            print(f"La computadora tendrá que realizar {2**n - 1} iteraciones complejas.")
            confirmacion = input("¿Deseas continuar de todos modos? (s/n): ")
            if confirmacion.lower() != 's':
                print("Ejecución cancelada.")
                return

        # Generar una matriz cuadrada aleatoria (del 1 al 10)
        matriz = np.random.randint(1, 11, size=(n, n))

        # Iniciar el temporizador
        tiempo_inicio = time.time()

        # Calcular el permanente usando nuestra implementación
        permanente = ryser_permanent(matriz)

        # Detener el temporizador
        tiempo_fin = time.time()
        tiempo_total = tiempo_fin - tiempo_inicio

        # Mostrar los resultados
        print("\n--- Resultados ---")
        print(f"Matriz generada ({n}x{n}):")
        print(matriz)
        print(f"\nPermanente de la matriz: {permanente:g}") 
        print(f"Tiempo de ejecución: {tiempo_total:.6f} segundos")

    except ValueError:
        print("Entrada no válida. Por favor, ingrese un número entero.")

# Ejecutar la función
calcular_permanente_aleatorio()

Ingrese el tamaño de la matriz cuadrada (ej. 3 para una de 3x3):  10



--- Resultados ---
Matriz generada (10x10):
[[ 5  5  7  2  3  3  7  6  9  8]
 [ 4  1  4  4  8  1  1  9  4  5]
 [ 3  1  9  9  4  8  6  6  5  1]
 [ 7  5  3  2 10  2  7  4  1  5]
 [ 6  1  2  2  9 10  6  6  8  4]
 [ 6  5  2  1  9  1  7  8  2  7]
 [ 3  4  3  5  6  7  9  5  2  6]
 [ 2  9  8  7  3  5  1  1  8  7]
 [ 8  9  8  6  3  5  2  4  2  8]
 [ 4  5  7  8  2  1  3  1  9  3]]

Permanente de la matriz: 3.44159e+13
Tiempo de ejecución: 0.027558 segundos
